# 3.6 — Multi-Agent Systems

A single agent handles one task at a time.
A **multi-agent system** splits complex tasks across specialised agents:

```
           User
            ↓
       Supervisor Agent
      (reads task, routes)
       /          \
 Researcher      Writer
 Agent           Agent
 (finds facts)   (writes content)
       \          /
        Final Answer
```

Benefits:
- Each agent is focused and has its own tools
- Tasks can be parallelised
- Easier to debug (isolate which agent failed)

In [ ]:
!pip install langchain langchain-ollama langgraph --quiet

## Step 1 — Define Worker Agents

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage

llm = ChatOllama(model='llama3.1', temperature=0)

# --- Researcher tools ---
@tool
def search_facts(topic: str) -> str:
    """Searches for key facts about a topic."""
    knowledge = {
        'python':     'Python was created by Guido van Rossum in 1991. It is dynamically typed, interpreted, and supports multiple paradigms.',
        'ai':         'Artificial Intelligence (AI) refers to machines that simulate human intelligence. Key fields: ML, NLP, Computer Vision.',
        'langchain':  'LangChain is an open-source framework for LLM applications. It provides tools, chains, agents, and memory abstractions.',
        'rag':        'RAG (Retrieval-Augmented Generation) improves LLM accuracy by retrieving relevant documents before generating a response.',
        'agents':     'LLM agents autonomously plan and execute multi-step tasks using tools. They follow a Thought-Action-Observation loop.',
    }
    for key, val in knowledge.items():
        if key in topic.lower():
            return val
    return f'No facts found for "{topic}".'

@tool
def get_stats(topic: str) -> str:
    """Retrieves statistics and numbers related to a topic."""
    stats = {
        'python':    'Python is used by 48% of developers. Over 8 million packages on PyPI. #1 language on GitHub.',
        'ai':        'Global AI market: $200B in 2023, projected $1.8 trillion by 2030. 77% of devices use AI.',
        'langchain': 'LangChain has 80k+ GitHub stars. Used in 100k+ projects. Supports 50+ LLM integrations.',
    }
    for key, val in stats.items():
        if key in topic.lower():
            return val
    return f'No statistics found for "{topic}".'

# --- Writer tools ---
@tool
def format_as_blog_post(title: str, content: str) -> str:
    """Formats content into a structured blog post with title, intro, and body."""
    return f"""# {title}

{content}

---
*Written by AI Writing Agent*"""

@tool
def format_as_bullet_points(content: str) -> str:
    """Reformats content into a concise bullet-point summary."""
    lines = [line.strip() for line in content.split('.') if line.strip()]
    bullets = '\n'.join(f'• {line}.' for line in lines[:5])
    return bullets

print('Worker tools defined.')

In [ ]:
# Worker agent runner — reusable for any agent
def run_worker(system_prompt: str, tools: list, task: str) -> str:
    """Run a worker agent with given tools and a task."""
    tool_map = {t.name: t for t in tools}
    llm_worker = llm.bind_tools(tools)

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=task)
    ]

    while True:
        response = llm_worker.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return response.content

        for tc in response.tool_calls:
            result = tool_map[tc['name']].invoke(tc['args'])
            print(f'    [{tc["name"]}] {str(result)[:80]}')
            messages.append(ToolMessage(content=str(result), tool_call_id=tc['id']))


RESEARCHER_PROMPT = """You are a research specialist.
Your job is to gather accurate facts and statistics about topics using your tools.
Return a comprehensive research summary."""

WRITER_PROMPT = """You are a professional content writer.
Your job is to take research notes and produce polished, readable content.
Use your formatting tools to structure the content appropriately."""

print('Worker agents ready.')

## Step 2 — Supervisor Agent

The supervisor reads the user request, decides which worker(s) to call, and assembles the final response.

In [ ]:
def supervisor_agent(user_request: str) -> str:
    """Orchestrate researcher and writer agents to fulfil a user request."""
    print(f'User Request: {user_request}')
    print('=' * 60)

    # Step 1: Ask supervisor which agents to invoke
    routing_prompt = f"""You are a supervisor managing a researcher and writer agent.
Given this user request, extract:
1. The TOPIC to research
2. The OUTPUT FORMAT requested (blog post / bullet points / summary)

User request: {user_request}

Reply with:
TOPIC: <topic>
FORMAT: <blog post|bullet points|summary>"""

    routing = llm.invoke([HumanMessage(content=routing_prompt)]).content
    print(f'Supervisor routing decision:\n{routing}\n')

    # Extract topic and format
    topic = 'AI'
    output_format = 'blog post'
    for line in routing.split('\n'):
        if line.startswith('TOPIC:'):
            topic = line.replace('TOPIC:', '').strip()
        if line.startswith('FORMAT:'):
            output_format = line.replace('FORMAT:', '').strip()

    # Step 2: Run researcher
    print(f'[Researcher Agent] researching: {topic}')
    research = run_worker(
        RESEARCHER_PROMPT,
        [search_facts, get_stats],
        f'Research everything you can find about: {topic}'
    )
    print(f'Research complete.\n')

    # Step 3: Run writer
    print(f'[Writer Agent] formatting as: {output_format}')
    writing_task = f"""Using this research, write a {output_format}:

Research notes:
{research}

Topic title: {topic}"""

    final_output = run_worker(
        WRITER_PROMPT,
        [format_as_blog_post, format_as_bullet_points],
        writing_task
    )

    print(f'\nFinal Output:')
    print('=' * 60)
    print(final_output)
    return final_output

print('Supervisor agent ready.')

In [ ]:
supervisor_agent('Write a blog post about Python programming')

In [ ]:
supervisor_agent('Give me bullet points about LangChain')

## Step 3 — Multi-Agent Patterns

| Pattern | Structure | Use case |
|---------|-----------|----------|
| **Supervisor** | One orchestrator, multiple workers | Complex tasks needing coordination |
| **Pipeline** | Agent A output → Agent B input | Sequential specialisation |
| **Parallel** | Multiple agents run simultaneously | Independent subtasks |
| **Debate** | Agents argue, one summarises | Getting diverse perspectives |

## Summary

- Worker agents are specialised — they have their own tools and system prompts
- The supervisor decides what work to delegate and in what order
- Output from one agent becomes input to the next
- Multi-agent systems are powerful for tasks too complex for a single agent